# 🔮 EducaStock — Previsão de Consumo por Produto (Prophet)

> **ML Caso de Uso 1** — ONG Casa da Criança  
> Modelo: [Prophet (Meta)](https://facebook.github.io/prophet/) para séries temporais  
> Alternativa de fallback: Média Móvel Ponderada  

## Fluxo
1. Autentica com Firebase Admin SDK
2. Exporta `stock_movements` do Firestore (tipo: saida)
3. Agrega consumo diário por produto
4. Treina um modelo Prophet por produto
5. Gera previsões semanais e mensais
6. Calcula sugestão de reposição
7. Escreve resultado na coleção `consumption_forecasts`

## Pré-requisitos
- Arquivo `serviceAccountKey.json` do Firebase Admin (upload na célula abaixo)
- Acesso ao projeto Firebase do EducaStock


In [ ]:
# ─── Instalar dependências ─────────────────────────────────────────────────
!pip install prophet firebase-admin pandas numpy matplotlib --quiet

In [ ]:
# ─── Upload do arquivo de credenciais Firebase ─────────────────────────────
from google.colab import files
import json

print("Faça upload do arquivo serviceAccountKey.json do Firebase Admin SDK")
print("Acesse: Firebase Console → Configurações do Projeto → Contas de Serviço → Gerar nova chave privada")
uploaded = files.upload()
SERVICE_ACCOUNT_KEY = list(uploaded.keys())[0]
print(f"\n✅ Arquivo carregado: {SERVICE_ACCOUNT_KEY}")

In [ ]:
# ─── Inicializar Firebase Admin ────────────────────────────────────────────
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_KEY)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase conectado com sucesso")

In [ ]:
# ─── Exportar movimentações de saída do Firestore ─────────────────────────
import pandas as pd
from datetime import datetime, timezone

def export_movements(limit: int = 5000) -> pd.DataFrame:
    """Exporta saídas de estoque e retorna DataFrame com colunas padronizadas."""
    print(f"Exportando até {limit} movimentações de saída...")

    docs = (
        db.collection("stock_movements")
        .where("type", "in", ["saida", "descarte"])
        .order_by("performedAt")
        .limit(limit)
        .stream()
    )

    records = []
    for doc in docs:
        d = doc.to_dict()
        try:
            performed_at = d.get("performedAt", "")
            if isinstance(performed_at, str):
                dt = datetime.fromisoformat(performed_at.replace("Z", "+00:00"))
            else:
                dt = performed_at  # Timestamp do Firestore

            records.append({
                "ds": dt.date(),  # coluna Prophet: data
                "product_id": d.get("productId", ""),
                "product_name": d.get("productName", "Desconhecido"),
                "quantity": int(d.get("quantity", 0)),
                "type": d.get("type", "saida"),
            })
        except Exception as e:
            print(f"  ⚠️  Registro ignorado: {e}")

    df = pd.DataFrame(records)
    print(f"✅ {len(df)} registros exportados")
    return df

df_raw = export_movements(limit=10000)

In [ ]:
# ─── Exportar estoque atual dos lotes ─────────────────────────────────────
def export_current_stock() -> dict:
    """Retorna {productId: total_quantity} com base nos lotes disponíveis."""
    print("Exportando estoque atual...")
    docs = (
        db.collection("batches")
        .where("status", "==", "available")
        .stream()
    )
    stock = {}
    for doc in docs:
        d = doc.to_dict()
        pid = d.get("productId", "")
        qty = int(d.get("quantity", 0))
        stock[pid] = stock.get(pid, 0) + qty

    print(f"✅ Estoque atual: {len(stock)} produtos")
    return stock

current_stock = export_current_stock()

In [ ]:
# ─── Pré-processamento ────────────────────────────────────────────────────
import numpy as np

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega consumo diário por produto."""
    if df.empty:
        return df

    df["ds"] = pd.to_datetime(df["ds"])
    daily = (
        df.groupby(["product_id", "product_name", "ds"])["quantity"]
        .sum()
        .reset_index()
        .rename(columns={"quantity": "y"})  # coluna Prophet: valor
    )
    return daily

df_daily = preprocess(df_raw)
print(f"Produtos com histórico de saída: {df_daily['product_id'].nunique()}")
df_daily.head(10)

In [ ]:
# ─── Modelo Prophet por produto ───────────────────────────────────────────
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")

MIN_DATA_POINTS = 10  # mínimo de dias com saída para treinar Prophet

def weighted_moving_average(series: pd.Series, window: int = 7) -> float:
    """Fallback: média móvel ponderada — pesos maiores para dados recentes."""
    if len(series) == 0:
        return 0.0
    n = min(len(series), window)
    recent = series.iloc[-n:].values
    weights = np.arange(1, n + 1, dtype=float)
    return float(np.dot(recent, weights) / weights.sum())

def compute_trend(forecast_df: pd.DataFrame) -> tuple[str, float]:
    """Calcula tendência comparando segunda quinzena vs. primeira."""
    if len(forecast_df) < 14:
        return "stable", 0.0
    first = forecast_df["yhat"].iloc[:15].mean()
    second = forecast_df["yhat"].iloc[15:].mean()
    if first == 0:
        return "stable", 0.0
    pct = ((second - first) / first) * 100
    if pct > 10:
        return "increasing", round(pct, 1)
    elif pct < -10:
        return "decreasing", round(abs(pct), 1)
    return "stable", round(abs(pct), 1)


def train_and_forecast(product_id: str, product_name: str,
                        df_product: pd.DataFrame) -> dict:
    """Treina Prophet ou usa fallback e retorna previsão."""
    n_points = len(df_product)

    if n_points < MIN_DATA_POINTS:
        # Fallback: média móvel ponderada
        daily_wma = weighted_moving_average(df_product["y"])
        weekly = round(daily_wma * 7, 2)
        monthly = round(daily_wma * 30, 2)
        stock = current_stock.get(product_id, 0)
        suggested = max(0, int(monthly * 1.2) - stock)
        return {
            "productId": product_id,
            "productName": product_name,
            "forecastWeekly": weekly,
            "forecastMonthly": monthly,
            "currentStock": stock,
            "suggestedReplenishment": suggested,
            "ciLower": None,
            "ciUpper": None,
            "trend": "stable",
            "trendPercent": 0.0,
            "modelVersion": "moving_average_v1",
            "generatedAt": datetime.now(timezone.utc).isoformat(),
            "dataPoints": n_points,
            "source": "moving_average",
        }

    # Prophet
    model = Prophet(
        interval_width=0.80,
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=(n_points >= 365),
        changepoint_prior_scale=0.05,
    )
    model.fit(df_product[["ds", "y"]])

    future = model.make_future_dataframe(periods=30)
    forecast = model.predict(future)

    future_only = forecast[forecast["ds"] > df_product["ds"].max()]
    weekly_rows = future_only.head(7)
    monthly_rows = future_only.head(30)

    forecast_weekly = max(0, weekly_rows["yhat"].sum())
    forecast_monthly = max(0, monthly_rows["yhat"].sum())
    ci_lower = max(0, weekly_rows["yhat_lower"].sum())
    ci_upper = max(0, weekly_rows["yhat_upper"].sum())

    trend, trend_pct = compute_trend(future_only)
    stock = current_stock.get(product_id, 0)
    suggested = max(0, int(forecast_monthly * 1.2) - stock)

    return {
        "productId": product_id,
        "productName": product_name,
        "forecastWeekly": round(forecast_weekly, 2),
        "forecastMonthly": round(forecast_monthly, 2),
        "currentStock": stock,
        "suggestedReplenishment": suggested,
        "ciLower": round(ci_lower, 2),
        "ciUpper": round(ci_upper, 2),
        "trend": trend,
        "trendPercent": trend_pct,
        "modelVersion": "prophet_v1",
        "generatedAt": datetime.now(timezone.utc).isoformat(),
        "dataPoints": n_points,
        "source": "prophet",
    }

print("✅ Funções de previsão definidas")

In [ ]:
# ─── Executar previsão para todos os produtos ─────────────────────────────
results = []
products = df_daily.groupby(["product_id", "product_name"])

print(f"Treinando modelos para {len(products)} produtos...\n")

for (pid, pname), group in products:
    group = group.sort_values("ds")
    try:
        result = train_and_forecast(pid, pname, group)
        results.append(result)
        model_tag = "Prophet" if result["source"] == "prophet" else "Média Móvel"
        print(f"  [{model_tag}] {pname[:40]}: {result['forecastWeekly']:.1f} un/sem | repor {result['suggestedReplenishment']} un.")
    except Exception as e:
        print(f"  ❌ Erro em {pname}: {e}")

print(f"\n✅ {len(results)} produtos processados")

In [ ]:
# ─── Visualização das principais previsões ────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_results = pd.DataFrame(results)

# Top 10 produtos que mais precisam de reposição
top_replenish = df_results[df_results["suggestedReplenishment"] > 0] \
    .sort_values("suggestedReplenishment", ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("EducaStock — Previsão de Consumo (Prophet)", fontsize=14, fontweight="bold")

# Gráfico 1: Sugestão de reposição
if not top_replenish.empty:
    colors = ["#C53030" if row["currentStock"] < row["forecastWeekly"] else "#B7791F"
              for _, row in top_replenish.iterrows()]
    axes[0].barh(top_replenish["productName"].str[:25], top_replenish["suggestedReplenishment"],
                 color=colors)
    axes[0].set_xlabel("Unidades a Repor")
    axes[0].set_title("Top Sugestões de Reposição")
    axes[0].invert_yaxis()

# Gráfico 2: Forecast mensal vs. estoque atual
sample = df_results.head(10)
x = range(len(sample))
axes[1].bar([i - 0.2 for i in x], sample["forecastMonthly"], width=0.4,
             label="Previsão 30d", color="#2F74C0", alpha=0.8)
axes[1].bar([i + 0.2 for i in x], sample["currentStock"], width=0.4,
             label="Estoque atual", color="#2E7D32", alpha=0.8)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([n[:12] for n in sample["productName"]], rotation=45, ha="right")
axes[1].set_ylabel("Unidades")
axes[1].set_title("Previsão 30d vs Estoque")
axes[1].legend()

plt.tight_layout()
plt.savefig("forecast_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Gráfico salvo: forecast_summary.png")

In [ ]:
# ─── Gravar previsões no Firestore ────────────────────────────────────────
from google.cloud.firestore_v1 import SERVER_TIMESTAMP

def write_forecasts_to_firestore(forecasts: list[dict]) -> None:
    col = db.collection("consumption_forecasts")
    batch = db.batch()
    count = 0

    for f in forecasts:
        doc_ref = col.document(f["productId"])
        data = {**f}
        batch.set(doc_ref, data, merge=False)
        count += 1

        # Firestore suporta no máximo 500 operações por batch
        if count % 400 == 0:
            batch.commit()
            batch = db.batch()
            print(f"  Batch enviado: {count} registros...")

    batch.commit()
    print(f"✅ {count} previsões gravadas em consumption_forecasts")

write_forecasts_to_firestore(results)

In [ ]:
# ─── Resumo final ────────────────────────────────────────────────────────
df_res = pd.DataFrame(results)

print("=" * 55)
print(" RESUMO — PREVISÃO DE CONSUMO EDUCASTOCK")
print("=" * 55)
print(f"Produtos processados     : {len(df_res)}")
print(f"Com modelo Prophet       : {(df_res['source'] == 'prophet').sum()}")
print(f"Com Média Móvel (fallback): {(df_res['source'] == 'moving_average').sum()}")
print(f"Precisam reposição       : {(df_res['suggestedReplenishment'] > 0).sum()}")
print(f"Consumo total previsto/mês: {df_res['forecastMonthly'].sum():.0f} unidades")
print("=" * 55)
print("\nTop 5 produtos para reposição urgente:")
urgent = df_res.nlargest(5, "suggestedReplenishment")[["productName", "suggestedReplenishment", "currentStock", "forecastMonthly", "source"]]
print(urgent.to_string(index=False))
print("\n✅ Previsões disponíveis no app EducaStock!")

## 📌 Instruções de uso

### Agendamento recomendado
Execute este notebook **1x por semana** ou após grandes movimentações de estoque.

### Índice Firestore necessário
Crie um índice composto no Firestore para a query:
```
Coleção: stock_movements
Campos: type (ASC), performedAt (ASC)
```

### Coleção gerada: `consumption_forecasts`
```json
{
  "productId": "abc123",
  "productName": "Arroz",
  "forecastWeekly": 5.2,
  "forecastMonthly": 21.0,
  "currentStock": 10,
  "suggestedReplenishment": 15,
  "ciLower": 3.5,
  "ciUpper": 7.0,
  "trend": "increasing",
  "trendPercent": 12.3,
  "modelVersion": "prophet_v1",
  "generatedAt": "2025-06-01T00:00:00Z",
  "dataPoints": 45,
  "source": "prophet"
}
```

### Boas práticas
- Nunca usar dados pessoais (usuário, endereço) no treinamento
- Manter baseline simples (média histórica) para comparar com Prophet
- Resultado é sempre **sugestão** — decisão final é do usuário
- Registrar previsões e feedback no Firestore para melhorar o modelo
